In [1]:
from google.colab import userdata
import os

aws_dir = os.path.expanduser("~/.aws")
os.makedirs(aws_dir, exist_ok=True)

key_id = userdata.get('AWS_ACCESS_KEY_ID')
secret_key = userdata.get('AWS_SECRET_ACCESS_KEY')

with open(f"{aws_dir}/credentials", "w") as f:
    f.write(f"[default]\naws_access_key_id = {key_id}\naws_secret_access_key = {secret_key}\n")

with open(f"{aws_dir}/config", "w") as f:
    f.write("[default]\nregion = us-east-1\n")

print("Successfully configured ~/.aws/credentials and ~/.aws/config")

Successfully configured ~/.aws/credentials and ~/.aws/config


In [2]:
!apt update -q && apt-get install -qy openjdk-21-jdk-headless > /dev/null 2>&1

import subprocess
from IPython.core.magic import register_cell_magic

class PersistentJShell:
    def __init__(self):
        self.proc = subprocess.Popen(
            ["jshell", "-s"],
            stdin=subprocess.PIPE,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1
        )

    def execute(self, code):
        if self.proc.poll() is not None:
            raise RuntimeError("JShell process has terminated.")

        marker = "___JSHELL_CELL_END___"
        payload = code.strip() + f'\nSystem.out.println("{marker}");\n'
        self.proc.stdin.write(payload)
        self.proc.stdin.flush()

        lines = []
        while True:
            line = self.proc.stdout.readline()
            if not line:
                break
            if marker in line:
                break
            lines.append(line)

        return "".join(lines)

_jshell_instance = PersistentJShell()

@register_cell_magic
def jshell(line, cell):
    out = _jshell_instance.execute(cell)
    if out:
        print(out, end="")


Get:1 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease [17.8 kB]
Hit:4 http://archive.ubuntu.com/ubuntu noble InRelease
Get:5 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:6 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:7 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ Packages [73.2 kB]
Get:8 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble/main amd64 Packages [52.1 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Get:11 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:12 http://security.ubuntu.com/ubuntu noble-security/main amd64 Packages [1,310 kB]
Get:13 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 Packages [1

In [3]:
%%jshell

void runPy(String code) throws Exception {
    String marker = "___END_OF_PYTHON_CELL___";

    // Base64 encode code payload to bypass all quote & newline syntax errors
    String b64 = java.util.Base64.getEncoder().encodeToString(code.getBytes(java.nio.charset.StandardCharsets.UTF_8));

    // Guarantee marker printing using try...finally and display trailing expressions using ast
    String payload = String.format(
        "import base64, ast, traceback\n" +
        "try:\n" +
        "    code_str = base64.b64decode('%s').decode('utf-8')\n" +
        "    tree = ast.parse(code_str)\n" +
        "    if tree.body and isinstance(tree.body[-1], ast.Expr):\n" +
        "        exec(compile(ast.Module(body=tree.body[:-1], type_ignores=[]), '<string>', 'exec'), globals())\n" +
        "        res = eval(compile(ast.Expression(body=tree.body[-1].value), '<string>', 'eval'), globals())\n" +
        "        if res is not None:\n" +
        "            print(res)\n" +
        "    else:\n" +
        "        exec(compile(tree, '<string>', 'exec'), globals())\n" +
        "except Exception:\n" +
        "    traceback.print_exc()\n" +
        "finally:\n" +
        "    print('%s', flush=True)\n",
        b64, marker
    );

    pyIn.println(payload);

    String line;
    while ((line = pyOut.readLine()) != null) {
        if (line.contains(marker)) break;
        System.out.println(line);
    }
}

import java.io.BufferedReader;
import java.io.InputStreamReader;
import java.io.PrintStream;
import java.io.IOException;

Process pythonProcess;
PrintStream pyIn;
BufferedReader pyOut;

try {
    ProcessBuilder pb = new ProcessBuilder("python3", "-i", "-u");
    // Redirect stderr to stdout to prevent pipe deadlock and capture Python errors
    pb.redirectErrorStream(true);
    pythonProcess = pb.start();

    pyIn = new PrintStream(pythonProcess.getOutputStream(), true);
    pyOut = new BufferedReader(new InputStreamReader(pythonProcess.getInputStream()));

    System.out.println("Python interpreter initialized safely.");
} catch (IOException e) {
    e.printStackTrace();
}

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Sep 30, 2026 8:40:21 AM java.util.prefs.FileSystemPreferences$1 run
INFO: Created user preferences directory.
->   ->  ->  ->  ->  ->  ->  ->  ->  ->  ->  ->  Python interpreter initialized safely.


# Lesson 7: vLLM SageMaker Realtime Deployment
This notebook demonstrates setting up AWS credentials via Colab Secrets, creating standard S3 buckets, and configuring a vLLM SageMaker Realtime Endpoint.

In [5]:
%%jshell

runPy("""
import os
# Cell 1 - Setup Environment & Repo
os.system('pip install boto3 sagemaker');

from pathlib import Path ; from IPython.display import display, HTML
import os

REPO_ROOT = Path('/content/BITS_programming')
if not REPO_ROOT.exists():
    os.system('git clone https://github.com/aqwertyuiop48/BITS_programming.git /content/BITS_programming');

NOTEBOOK_DIR = REPO_ROOT / 'module_5/week_27'
os.chdir(NOTEBOOK_DIR)
print(f'Working directory: {NOTEBOOK_DIR}')
""");

->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Collecting boto3
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.2/44.2 kB 2.7 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of aiobotocore to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of aiobotocore to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
INFO: pip is looking at multiple versions of s3fs to determine which version is compatible with other requirements. This could take

In [6]:
%%jshell

runPy("""
# Cell 2 - Retrieve AWS Credentials safely via AWS Config / Colab Secrets / Env
import os, pathlib, configparser

def get_aws_config_cred(profile="default"):
    cred_file = pathlib.Path.home() / ".aws" / "credentials"
    config_file = pathlib.Path.home() / ".aws" / "config"

    parser = configparser.ConfigParser()
    parser.read([cred_file, config_file])

    profile_section = profile if profile in parser else f"profile {profile}"

    if profile_section in parser:
        sec = parser[profile_section]
        key_id = sec.get("aws_access_key_id")
        secret_key = sec.get("aws_secret_access_key")
        session_token = sec.get("aws_session_token")
        if key_id and secret_key:
            return key_id, secret_key, session_token

    return None, None, None

def get_colab_secret(*names, required=True):
    for name in names:
        value = os.environ.get(name)
        if value:
            return value

    try:
        from google.colab import userdata
    except ImportError:
        userdata = None

    if userdata is not None:
        for name in names:
            try:
                value = os.environ.get(name) or userdata.get(name)
                if value:
                    return value
            except Exception:
                continue

    if required:
        choices = " or ".join(names)
        raise RuntimeError(
            f"Missing AWS credential. Configure ~/.aws/config or ~/.aws/credentials, "
            f"add a Colab Secret named {choices}, or set environment variables."
        )
    return None

# 1. Try reading from AWS config/credentials file
aws_profile = os.environ.get("AWS_PROFILE", "default")
file_key, file_secret, file_token = get_aws_config_cred(aws_profile)

# 2. Fall back to Colab secrets or environment variables
AWS_ACCESS_KEY_ID = file_key or get_colab_secret('AWS_ACCESS_KEY_ID', 'AWS_ACCESS_KEY')
AWS_SECRET_ACCESS_KEY = file_secret or get_colab_secret('AWS_SECRET_ACCESS_KEY', 'AWS_SECRET_KEY')
AWS_SESSION_TOKEN = file_token or get_colab_secret('AWS_SESSION_TOKEN', 'AWS_SECURITY_TOKEN', required=False)

os.environ['AWS_ACCESS_KEY_ID'] = AWS_ACCESS_KEY_ID
os.environ['AWS_SECRET_ACCESS_KEY'] = AWS_SECRET_ACCESS_KEY
if AWS_SESSION_TOKEN:
    os.environ['AWS_SESSION_TOKEN'] = AWS_SESSION_TOKEN

source = f"~/.aws/config or credentials [profile: {aws_profile}]" if file_key else "Colab secrets / Environment variables"
print(f"AWS credentials loaded from {source} and set as environment variables.")
""");

->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... AWS credentials loaded from ~/.aws/config or credentials [profile: default] and set as environment variables.


In [8]:
%%jshell

runPy("""
# Block 1 - Config & S3 Bucket Naming
import boto3
import datetime as dt
from botocore.exceptions import ClientError

AWS_REGION_NAME     = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")
ALLOW_AWS_MUTATIONS = True

PROJECT_NAME  = "lesson7-vllm"
ENVIRONMENT   = "demo"
HF_MODEL_ID   = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
NUM_GPUS      = 1

os.environ["AWS_DEFAULT_REGION"] = AWS_REGION_NAME
os.environ["AWS_REGION"]         = AWS_REGION_NAME
region       = AWS_REGION_NAME

boto_session = boto3.Session(region_name=region)
sts = boto_session.client("sts")
_ident = sts.get_caller_identity()
account_id = _ident["Account"]
caller_arn = _ident["Arn"]

s3  = boto_session.client("s3")
sm  = boto_session.client("sagemaker")
smr = boto_session.client("sagemaker-runtime")
iam = boto_session.client("iam")
cw  = boto_session.client("cloudwatch")

# Dynamic S3 Bucket setup following standardization
S3_BUCKET_NAME = f"{PROJECT_NAME}-{account_id}-{region}"
MODEL_S3_URI  = f"s3://{S3_BUCKET_NAME}/{PROJECT_NAME}/model/"

# Ensure standard bucket exists
try:
    if region == "us-east-1":
        s3.create_bucket(Bucket=S3_BUCKET_NAME)
    else:
        s3.create_bucket(
            Bucket=S3_BUCKET_NAME,
            CreateBucketConfiguration={'LocationConstraint': region}
        )
    print(f"Target S3 Bucket ready: s3://{S3_BUCKET_NAME}")
except ClientError as e:
    if e.response['Error']['Code'] in ['BucketAlreadyOwnedByYou', 'BucketAlreadyExists']:
        print(f"Using existing S3 Bucket: s3://{S3_BUCKET_NAME}")
    else:
        raise e

GPU_INSTANCE_CANDIDATES = [
    "ml.g4dn.xlarge",
    "ml.g5.xlarge",
    "ml.g4dn.2xlarge",
    "ml.g5.2xlarge",
    "ml.p3.2xlarge",
]

INSTANCE_HOURLY = {
    "ml.g4dn.xlarge": 0.736, "ml.g5.xlarge": 1.408, "ml.g4dn.2xlarge": 0.94,
    "ml.g5.2xlarge": 1.515, "ml.p3.2xlarge": 3.825,
}

timestamp       = dt.datetime.utcnow().strftime("%Y%m%d-%H%M%S")
model_name      = f"{PROJECT_NAME}-{ENVIRONMENT}-model-{timestamp}"
endpoint_name   = f"{PROJECT_NAME}-{ENVIRONMENT}-ep"
endpoint_config = None
chosen_instance = None

print(f"Region:     {region}")
print(f"Account:    {account_id}")
print(f"Model:      {HF_MODEL_ID}")
print(f"S3 URI:     {MODEL_S3_URI}")
print(f"Sweep:      {GPU_INSTANCE_CANDIDATES}")
print(f"Endpoint:   {endpoint_name}")
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Target S3 Bucket ready: s3://lesson7-vllm-455865672536-us-east-1
Region:     us-east-1
Account:    455865672536
Model:      TinyLlama/TinyLlama-1.1B-Chat-v1.0
S3 URI:     s3://lesson7-vllm-455865672536-us-east-1/lesson7-vllm/model/
Sweep:      ['ml.g4dn.xlarge', 'ml.g5.xlarge', 'ml.g4dn.2xlarge', 'ml.g5.2xlarge', 'ml.p3.2xlarge']
Endpoint:   lesson7-vllm-demo-ep


In [9]:
%%jshell

runPy("""
import datetime, pytz;
print("Current Time in IST:", datetime.datetime.now(pytz.utc).astimezone(pytz.timezone('Asia/Kolkata')).strftime('%Y-%m-%d %H:%M:%S'))
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Current Time in IST: 2026-09-30 14:15:31
